In [2]:
import pandas as pd
import time
from geopy.geocoders import Nominatim

# 1. Load the original dataset
input_file = "final_landmarks_dataset.csv"
df = pd.read_csv(input_file)

print(f"Total rows in dataset: {len(df)}")

# Ensure columns are numeric so we can accurately spot missing ones
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# Find how many are missing
missing_mask = df['Latitude'].isna() | df['Longitude'].isna()
missing_count = missing_mask.sum()
print(f"Rows missing coordinates: {missing_count}")

# 2. Setup the free geocoder
geolocator = Nominatim(user_agent="algiers_rescue_mission")

def rescue_coordinates(name):
    try:
        # Add "Algiers, Algeria" so the search engine knows exactly where to look
        query = f"{name}, Algiers, Algeria"
        location = geolocator.geocode(query, timeout=10)
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        pass # Ignore connection errors and keep going
    return None, None

# 3. Try to rescue the missing ones
print(f"\nAttempting to rescue {missing_count} locations. This might take a few minutes...")
rescued_count = 0

for index in df[missing_mask].index:
    # Use row['name'] or row['Name'] depending on your exact column capitalization
    name_col = 'name' if 'name' in df.columns else 'Name'
    name = df.at[index, name_col]
    
    # Skip if the name itself is somehow empty
    if pd.isna(name):
        continue
        
    lat, lon = rescue_coordinates(name)
    
    if lat and lon:
        df.at[index, 'Latitude'] = lat
        df.at[index, 'Longitude'] = lon
        rescued_count += 1
        print(f" [Rescued] {name} -> {lat}, {lon}")
    
    # Sleep to respect OpenStreetMap's free API limits (1 request per second max)
    time.sleep(1.2)

# 4. Final tally and save
still_missing = df['Latitude'].isna() | df['Longitude'].isna()
print(f"\nRescue complete! We recovered {rescued_count} locations.")
print(f"Rows that are STILL missing (unfindable): {still_missing.sum()}")

# NOW we drop the truly unfindable ones, which should be a much smaller number
df_cleaned = df.dropna(subset=['Latitude', 'Longitude']).copy()
df_cleaned.reset_index(drop=True, inplace=True)

print(f"Final valid rows ready for AI: {len(df_cleaned)}")

# Save this rescued dataset so we don't have to do it again
df_cleaned.to_csv("final_landmarks.csv", index=False, encoding='utf-8-sig')
print("Saved clean data to 'rescued_landmarks.csv'")

Total rows in dataset: 724
Rows missing coordinates: 552

Attempting to rescue 552 locations. This might take a few minutes...

Rescue complete! We recovered 0 locations.
Rows that are STILL missing (unfindable): 552
Final valid rows ready for AI: 172
Saved clean data to 'rescued_landmarks.csv'


In [4]:
import pandas as pd
import numpy as np

# Load the dataset
input_file = "final_landmarks.csv"
df = pd.read_csv(input_file)

print(f"Rows before cleaning: {len(df)}")

# Force Latitude and Longitude to be numeric. Any empty strings or garbage text becomes NaN (Not a Number)


print(f"Rows after cleaning missing coordinates: {len(df)}")

Rows before cleaning: 172
Rows after cleaning missing coordinates: 172


In [5]:
import requests
import json
import time

# --- ADD YOUR GROQ API KEYS HERE ---
GROQ_API_KEYS = [
    "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", 
    "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",
    "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" ,
    "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"
]

current_key_index = 0

def get_next_key():
    """Rotates through the list of API keys."""
    global current_key_index
    key = GROQ_API_KEYS[current_key_index]
    # Move to the next key for the next request
    current_key_index = (current_key_index + 1) % len(GROQ_API_KEYS)
    return key

def fetch_landmark_data_from_ai(landmark_name):
    """Asks Groq to generate a description, category, and rating in JSON format."""
    
    prompt = f"""
    You are an expert Algerian tour guide. I will give you the name of a landmark in Algiers, Algeria.
    Provide the following information in strict JSON format:
    1. "description": A concise, accurate 2-sentence description of the place.
    2. "category": Choose the most accurate category (e.g., Museum, Historic Mosque, Park, Monument, Beach, Local Attraction).
    3. "rating": An estimated Google Maps rating out of 10.0 (e.g., 4.5) based on its popularity and fame.

    Landmark Name: {landmark_name}
    
    Respond ONLY with the JSON object.
    """
    
    # We use Llama 3 8B because it is exceptionally fast and has generous free rate limits on Groq
    payload = {
        "model": "llama-3.1-8b-instant", 
        "messages": [{"role": "user", "content": prompt}],
        "response_format": {"type": "json_object"},
        "temperature": 0.2 # Keep it factual
    }
    
    # Retry logic in case of rate limits
    max_retries = 3
    for attempt in range(max_retries):
        headers = {
            "Authorization": f"Bearer {get_next_key()}",
            "Content-Type": "application/json"
        }
        
        try:
            response = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
            
            if response.status_code == 200:
                content = response.json()['choices'][0]['message']['content']
                return json.loads(content)
            elif response.status_code == 429: # Rate Limit Exceeded
                print(f"Rate limited! Switching key and waiting 5 seconds... (Attempt {attempt+1}/{max_retries})")
                time.sleep(5) 
            else:
                print(f"API Error {response.status_code}: {response.text}")
                break
                
        except Exception as e:
            print(f"Request failed: {e}")
            time.sleep(2)
            
    return {"description": "Information not found", "category": "Other", "rating": "N/A"}

# Add the new Rating column if it doesn't exist
if 'Rating' not in df.columns:
    df['Rating'] = None

print("Starting AI Enrichment...")

for index, row in df.iterrows():
    name = row['name']
    
    # Only fetch if description is missing, empty, or "No description"
    if pd.isna(row.get('Description')) or "No description" in str(row.get('Description')):
        print(f"Processing ({index+1}/{len(df)}): {name}...")
        
        ai_data = fetch_landmark_data_from_ai(name)
        
        df.at[index, 'Description'] = ai_data.get('description', "No description")
        df.at[index, 'Category'] = ai_data.get('category', "Other")
        df.at[index, 'Rating'] = ai_data.get('rating', "N/A")
        
        # Sleep for 2 seconds between requests to respect free tier quotas
        time.sleep(2) 

print("AI Enrichment Complete!")

Starting AI Enrichment...
Processing (1/172): Martyrs Memorial...
Processing (3/172): Port of Algiers...
Processing (6/172): Bastion 23...
Processing (8/172): Sablette...
Processing (9/172): Teri park...
Processing (11/172): Kiffan Club...
Processing (14/172): Emir Abdelkader Place...
Processing (18/172): Bab Azzoun...
Processing (19/172): Tennis Bachdjerrah...
Processing (20/172): Bois Des Cars...
Processing (21/172): Beyrouth Parc...
Processing (23/172): Dounia Parc...
Processing (26/172): Port Saïd Square...
Processing (27/172): International Conference Center Abdelatif Rahal...
Processing (29/172): Tifariti Garden...
Processing (31/172): Ali la Pointe Museum...
Processing (34/172): La Vigie Alger...
Processing (36/172): Prague Garden...
Processing (38/172): Ferhani Stadium...
Processing (39/172): Musée de Tamentfoust - Le Fort Turc...
Processing (42/172): Stadium Baba Hassen...
Processing (43/172): Olof Palme Garden...
Processing (44/172): Bazita...
Processing (45/172): Ben Aknoun 

In [6]:
import requests

def get_wikipedia_image(landmark_name):
    # Search both English and French Wikipedia as Algiers data is heavily in French
    for lang in ['en', 'fr']:
        api_url = f"https://{lang}.wikipedia.org/w/api.php"
        search_params = {
            "action": "query", "list": "search",
            "srsearch": f"{landmark_name} Alger", 
            "format": "json", "srlimit": 1
        }
        
        try:
            search_res = requests.get(api_url, params=search_params).json()
            search_results = search_res.get('query', {}).get('search', [])
            
            if search_results:
                best_match_title = search_results[0]['title']
                details_params = {
                    "action": "query", "titles": best_match_title,
                    "prop": "pageimages", "pithumbsize": 800, "format": "json"
                }
                
                details_res = requests.get(api_url, params=details_params).json()
                pages = details_res.get('query', {}).get('pages', {})
                
                for page_id, page_info in pages.items():
                    if 'thumbnail' in page_info:
                        return page_info['thumbnail']['source']
        except Exception:
            continue
            
    return "No image available"

print("Fetching images...")
for index, row in df.iterrows():
    if pd.isna(row.get('Image_URL')) or "No image" in str(row.get('Image_URL')):
        df.at[index, 'Image_URL'] = get_wikipedia_image(row['name'])

print("Image fetching complete!")

Fetching images...
Image fetching complete!


In [7]:
output_file = "algiers_landmarks_final_version.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Done! Your AI-enriched dataset is saved as: {output_file}")
print(df.head())

Done! Your AI-enriched dataset is saved as: algiers_landmarks_final_version.csv
                      name   Latitude  Longitude  \
0         Martyrs Memorial  36.745712   3.069732   
1   Botanical Garden Hamma  36.746676   3.072736   
2          Port of Algiers  36.763647   3.060971   
3  Great Mosque of Algeria  36.785389   3.063927   
4          Martyrs' Square  36.784749   3.062351   

                                         Description           Image_URL  \
0  The Martyrs Memorial is a monument in Algiers,...  No image available   
1  The Botanical Garden Hamma (Arabic: حديقة التج...  No image available   
2  The Port of Algiers is a major seaport in Alge...  No image available   
3  The Djamaa el Djazaïr (Arabic: جامع الجزائر), ...  No image available   
4                      Martyrs' Square may refer to:  No image available   

         Category Rating  
0        Monument    4.2  
1  Other Landmark   None  
2            Port    4.2  
3          Mosque   None  
4  Other Landma